# VQA Yes/No — Runbook

Trains the attention-based yes/no VQA model on VQA v2.

**Before running:** Make sure you're connected to a GPU runtime.

**Steps:**
1. Install dependencies
2. Mount Drive (persists checkpoints)
3. Download VQA v2 annotations
4. Download COCO images (only the ones needed)
5. Build vocabulary
6. Train
7. Evaluate
8. Single-image inference

## 1. Install Dependencies

In [ ]:
import os, sys

if not os.path.isdir('/content/Neural_Image_Caption_Generator'):
    !git clone -b experiment/visualQA https://github.com/Miwi343/Neural_Image_Caption_Generator.git /content/Neural_Image_Caption_Generator
else:
    !git -C /content/Neural_Image_Caption_Generator pull

REPO = '/content/Neural_Image_Caption_Generator'
if REPO not in sys.path:
    sys.path.insert(0, REPO)

os.chdir(REPO)

!pip install -q -r requirements.txt


In [ ]:
import torch
print("torch       :", torch.__version__)
print("cuda        :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu         :", torch.cuda.get_device_name(0))
    print("gpu memory  :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

## 2. Mount Drive

Checkpoints and results are symlinked into Drive so they survive disconnects.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, shutil
from pathlib import Path

DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/vqa_results")
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def link_output_dir(local_name: str):
    local  = Path(local_name)
    remote = DRIVE_OUTPUT_DIR / local_name
    remote.mkdir(parents=True, exist_ok=True)
    if local.is_symlink():
        print(f"Already linked: {local}")
        return
    if local.exists():
        contents = list(local.iterdir())
        if len(contents) == 0 or (len(contents) == 1 and contents[0].name == ".gitkeep"):
            shutil.rmtree(local)
        else:
            print(f"Using existing local {local}")
            return
    os.symlink(remote, local)
    print(f"Linked: {local} -> {remote}")

link_output_dir("checkpoints_vqa")
link_output_dir("results_vqa")

## 3. Download VQA v2 Annotations

Downloads 4 JSON files (~80 MB total) directly from visualqa.org. No Kaggle needed.

In [ ]:
import subprocess, zipfile

DATA_ROOT = Path("data/vqa")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

VQA_DOWNLOADS = [
    ("https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Annotations_Train_mscoco.zip",  "v2_mscoco_train2014_annotations.json"),
    ("https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Annotations_Val_mscoco.zip",    "v2_mscoco_val2014_annotations.json"),
    ("https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Questions_Train_mscoco.zip",    "v2_OpenEnded_mscoco_train2014_questions.json"),
    ("https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Questions_Val_mscoco.zip",      "v2_OpenEnded_mscoco_val2014_questions.json"),
]

for url, expected_json in VQA_DOWNLOADS:
    dest = DATA_ROOT / expected_json
    if dest.exists():
        print(f"Already exists: {expected_json}")
        continue
    zip_dest = DATA_ROOT / url.split("/")[-1]
    print(f"Downloading {url.split('/')[-1]} …")
    subprocess.run(["wget", "-q", "-O", str(zip_dest), url], check=True)
    with zipfile.ZipFile(zip_dest) as z:
        z.extractall(DATA_ROOT)
    zip_dest.unlink()
    print(f"  → {expected_json}")

print("\nAnnotation files:")
for f in sorted(DATA_ROOT.glob("*.json")):
    print(f"  {f.name}  ({f.stat().st_size // 1_000_000} MB)")

## 4. Download COCO Images

Images are stored persistently on Drive (`MyDrive/vqa_results/images/`) and symlinked into the repo. On the first run this downloads ~80k yes/no images (~8 GB, 20-40 min). Every subsequent session skips the download entirely.

In [ ]:
import json, urllib.request
from concurrent.futures import ThreadPoolExecutor, as_completed

# Images live on Drive so they persist across sessions — no re-download needed.
IMG_DIR = Path("/content/drive/MyDrive/vqa_results/images")
(IMG_DIR / "train2014").mkdir(parents=True, exist_ok=True)
(IMG_DIR / "val2014").mkdir(parents=True, exist_ok=True)

# Symlink into the repo's expected path so training code finds them
LOCAL_IMG_DIR = DATA_ROOT / "images"
if not LOCAL_IMG_DIR.exists() and not LOCAL_IMG_DIR.is_symlink():
    os.symlink(IMG_DIR, LOCAL_IMG_DIR)
    print(f"Linked: {LOCAL_IMG_DIR} -> {IMG_DIR}")

def get_needed_image_ids(data_root):
    needed = {"train2014": set(), "val2014": set()}
    for coco_split in ("train2014", "val2014"):
        with open(data_root / f"v2_mscoco_{coco_split}_annotations.json") as f:
            for ann in json.load(f)["annotations"]:
                if ann["answer_type"] == "yes/no":
                    needed[coco_split].add(ann["image_id"])
    return needed

def download_image(image_id, coco_split, img_dir):
    filename = f"COCO_{coco_split}_{image_id:012d}.jpg"
    dest = img_dir / coco_split / filename
    if dest.exists():
        return "skip"
    url = f"http://images.cocodataset.org/{coco_split}/{filename}"
    try:
        urllib.request.urlretrieve(url, dest)
        return "ok"
    except Exception as e:
        return f"err:{e}"

print("Finding yes/no image IDs …")
needed = get_needed_image_ids(DATA_ROOT)
for split, ids in needed.items():
    print(f"  {split}: {len(ids):,} images needed")

total = sum(len(v) for v in needed.values())
already = sum(
    1 for coco_split, ids in needed.items()
    for img_id in ids
    if (IMG_DIR / coco_split / f"COCO_{coco_split}_{img_id:012d}.jpg").exists()
)
print(f"\n{already:,} already downloaded, {total - already:,} to go.")

if already < total:
    print("Downloading (16 threads) — this takes 20-40 min on first run …\n")
    done, errors = 0, []
    with ThreadPoolExecutor(max_workers=16) as pool:
        futures = {
            pool.submit(download_image, img_id, coco_split, IMG_DIR): (img_id, coco_split)
            for coco_split, ids in needed.items() for img_id in ids
        }
        for future in as_completed(futures):
            r = future.result()
            done += 1
            if r.startswith("err"):
                errors.append(r)
            if done % 2000 == 0:
                print(f"  {done}/{total} ({done/total:.0%}) — {len(errors)} errors")
    print(f"\nDone. {done-len(errors):,} downloaded, {len(errors)} errors.")
else:
    print("All images already present on Drive — skipping download.")

## 5. Build Vocabulary

In [ ]:
import sys, os
from pathlib import Path

REPO = '/content/Neural_Image_Caption_Generator'
if not os.path.isdir(REPO):
    import subprocess
    subprocess.run(["git", "clone", "https://github.com/Miwi343/Neural_Image_Caption_Generator.git", REPO], check=True)
if REPO not in sys.path:
    sys.path.insert(0, REPO)
os.chdir(REPO)

DATA_ROOT = Path("data/vqa")

from vqa.dataset import QuestionVocabulary, build_and_save_vocab
from vqa.config import VOCAB_PATH, VOCAB_SIZE

vocab_path = Path(VOCAB_PATH)
if vocab_path.exists():
    vocab = QuestionVocabulary.load(str(vocab_path))
    print(f'Loaded vocab: {len(vocab):,} tokens')
else:
    vocab = build_and_save_vocab(str(DATA_ROOT), str(vocab_path), max_size=VOCAB_SIZE)

# Backup to Drive
!mkdir -p "/content/drive/MyDrive/vqa_results/data/vqa"
!cp data/vqa/vocab.json "/content/drive/MyDrive/vqa_results/data/vqa/vocab.json"


### Dataset sanity check

In [ ]:
from vqa.dataset import VQAYesNoDataset

ds = VQAYesNoDataset(str(DATA_ROOT), vocab, split="val", max_samples=2000)
labels = [ds.pairs[i][2] for i in range(len(ds))]
yes_count = sum(labels)
print(f"Val samples : {len(ds):,}")
print(f"Yes: {yes_count:,}  No: {len(labels)-yes_count:,}  ({yes_count/len(labels):.1%} yes)")
print("\nSample questions:")
for i in range(5):
    img_id, q, label = ds.pairs[i]
    print(f"  {q!r}  → {'yes' if label else 'no'}")

## 6. Train

In [ ]:

# Copy images from Drive to local Colab SSD (one-time per session, ~3-5 min).
# Local SSD is 20-50x faster than Drive for random reads, which is the
# dominant bottleneck during training.
import os, shutil
from pathlib import Path

DRIVE_IMG  = Path("/content/drive/MyDrive/vqa_results/images")
LOCAL_IMG  = Path("/content/images_local")
LOCAL_LINK = Path("data/vqa/images")

if LOCAL_IMG.exists():
    print("Local image cache already present — skipping copy.")
else:
    print("Copying images from Drive to local SSD …")
    shutil.copytree(str(DRIVE_IMG), str(LOCAL_IMG))
    print(f"  Done. {sum(1 for _ in LOCAL_IMG.rglob('*.jpg')):,} images copied.")

# Re-point the repo symlink to local SSD
if LOCAL_LINK.is_symlink() or LOCAL_LINK.exists():
    if LOCAL_LINK.is_symlink():
        LOCAL_LINK.unlink()
    else:
        shutil.rmtree(str(LOCAL_LINK))
os.symlink(str(LOCAL_IMG), str(LOCAL_LINK))
print(f"Symlink: {LOCAL_LINK} → {LOCAL_IMG}")


In [ ]:
!PYTHONPATH=/content/Neural_Image_Caption_Generator python -m vqa.train


In [ ]:
import csv, matplotlib.pyplot as plt

log_path = Path("results_vqa/training_log.csv")
if log_path.exists():
    with open(log_path) as f:
        rows = list(csv.DictReader(f))
    epochs     = [int(r["epoch"])         for r in rows]
    train_acc  = [float(r["train_acc"])   for r in rows]
    val_acc    = [float(r["val_acc"])     for r in rows]
    train_loss = [float(r["train_loss"])  for r in rows]
    val_loss   = [float(r["val_loss"])    for r in rows]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(epochs, train_acc, label="train"); ax1.plot(epochs, val_acc, label="val")
    ax1.set_title("Accuracy"); ax1.set_xlabel("epoch"); ax1.legend(); ax1.grid(alpha=0.3)
    ax2.plot(epochs, train_loss, label="train"); ax2.plot(epochs, val_loss, label="val")
    ax2.set_title("Loss"); ax2.set_xlabel("epoch"); ax2.legend(); ax2.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig("results_vqa/training_curves.png", dpi=150)
    plt.show()
else:
    print("No log yet.")

## 7. Evaluate

In [ ]:
!PYTHONPATH=/content/Neural_Image_Caption_Generator python -m vqa.evaluate \
    --checkpoint checkpoints_vqa/best.pt \
    --data_root data/vqa \
    --vocab_path data/vqa/vocab.json \
    --batch_size 64


## 8. Hyperparameter Sweep

Runs a small grid of configs after the vanilla baseline and picks the best one.
Each run saves its own checkpoint and training log under a tagged subdirectory
(e.g. `checkpoints_vqa_lr1e3/`, `results_vqa_lr1e3/`).
A comparison table and plot are produced at the end.

In [ ]:
import sys, os
from pathlib import Path

REPO = '/content/Neural_Image_Caption_Generator'
if REPO not in sys.path:
    sys.path.insert(0, REPO)
os.chdir(REPO)

from vqa.train import main as train_main

# ------------------------------------------------------------------
# Sweep configs — each dict overrides defaults from vqa/config.py.
# 'tag' controls the output subdirectory name.
# ------------------------------------------------------------------
SWEEP_CONFIGS = [
    # Higher LR
    {"lr": 1e-3, "encoder_lr": 3e-4, "tag": "lr1e3"},
    # Lower LR
    {"lr": 1e-4, "encoder_lr": 3e-5, "tag": "lr1e4"},
    # Less dropout (less regularisation)
    {"dropout": 0.3, "tag": "drop03"},
    # More dropout (more regularisation)
    {"dropout": 0.7, "tag": "drop07"},
    # Larger batch
    {"batch_size": 256, "tag": "bs256"},
]

sweep_results = []
for cfg in SWEEP_CONFIGS:
    best_acc = train_main(cfg)
    sweep_results.append({"tag": cfg["tag"], "cfg": cfg, "best_val_acc": best_acc})
    print(f"[sweep] {cfg['tag']:12s}  best_val_acc={best_acc:.4f}")

print("\nSweep complete.")

In [ ]:
import csv
from pathlib import Path

# Load vanilla baseline result for comparison
vanilla_log = Path('results_vqa/training_log.csv')
vanilla_acc = 0.0
if vanilla_log.exists():
    with open(vanilla_log) as f:
        rows = list(csv.DictReader(f))
    vanilla_acc = max(float(r['val_acc']) for r in rows)

all_results = [{"tag": "vanilla", "cfg": {}, "best_val_acc": vanilla_acc}] + sweep_results

print(f"{'Run':<14} {'lr':>8} {'enc_lr':>8} {'dropout':>8} {'batch':>6}  best_val_acc")
print("-" * 65)
for r in sorted(all_results, key=lambda x: -x['best_val_acc']):
    c = r['cfg']
    print(
        f"{r['tag']:<14} "
        f"{c.get('lr', 3e-4):>8.1e} "
        f"{c.get('encoder_lr', 1e-4):>8.1e} "
        f"{c.get('dropout', 0.5):>8.2f} "
        f"{c.get('batch_size', 128):>6}  "
        f"{r['best_val_acc']:.4f}"
    )

best_run = max(all_results, key=lambda x: x['best_val_acc'])
print(f"\nBest run: {best_run['tag']}  ({best_run['best_val_acc']:.4f} val acc)")

In [ ]:
import csv, matplotlib.pyplot as plt
from pathlib import Path

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left: val acc curves for every run ---
for r in all_results:
    tag = r['tag']
    log = Path(f"results_vqa{'_' + tag if tag != 'vanilla' else ''}/training_log.csv")
    if not log.exists():
        continue
    with open(log) as f:
        rows = list(csv.DictReader(f))
    epochs  = [int(row['epoch'])       for row in rows]
    val_acc = [float(row['val_acc'])   for row in rows]
    lw = 2.5 if tag == best_run['tag'] else 1.0
    axes[0].plot(epochs, val_acc, label=tag, linewidth=lw)

axes[0].set_title('Val Accuracy — all runs')
axes[0].set_xlabel('epoch')
axes[0].set_ylabel('val acc')
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

# --- Right: bar chart of best val acc per run ---
tags = [r['tag'] for r in all_results]
accs = [r['best_val_acc'] for r in all_results]
colors = ['steelblue' if t != best_run['tag'] else 'tomato' for t in tags]
bars = axes[1].bar(tags, accs, color=colors)
axes[1].set_title('Best Val Accuracy per Run')
axes[1].set_ylabel('best val acc')
axes[1].set_ylim(min(accs) - 0.02, max(accs) + 0.02)
axes[1].tick_params(axis='x', rotation=30)
for bar, acc in zip(bars, accs):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
                 f'{acc:.3f}', ha='center', va='bottom', fontsize=8)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('results_vqa/sweep_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results_vqa/sweep_comparison.png')

## 9. Single-Image Inference

In [ ]:
from google.colab import files
uploaded = files.upload()
IMAGE_PATH = next(iter(uploaded))
print("Using image:", IMAGE_PATH)

In [ ]:
QUESTION = "Is there a dog in this image?"  # <-- change this

import sys
REPO = '/content/Neural_Image_Caption_Generator'
if REPO not in sys.path:
    sys.path.insert(0, REPO)

from vqa.evaluate import answer_question
result = answer_question(
    checkpoint_path='checkpoints_vqa/best.pt',
    image_path=IMAGE_PATH,
    question=QUESTION,
    vocab_path='data/vqa/vocab.json',
)
print(f"Answer     : {result['answer'].upper()}")
print(f"Confidence : {result['confidence']:.1%}")


In [ ]:
import numpy as np, matplotlib.pyplot as plt
from PIL import Image
from scipy.ndimage import gaussian_filter

img = np.array(Image.open(IMAGE_PATH).convert("RGB").resize((224, 224)))
alpha = result["alpha"].reshape(14, 14)
alpha_up = np.kron(alpha, np.ones((16, 16)))
alpha_smooth = gaussian_filter(alpha_up, sigma=8)
alpha_norm = (alpha_smooth - alpha_smooth.min()) / (alpha_smooth.max() - alpha_smooth.min() + 1e-8)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(img); axes[0].set_title("Input"); axes[0].axis("off")
axes[1].imshow(img); axes[1].imshow(alpha_norm, alpha=0.55, cmap="hot")
axes[1].set_title(f"A: {result['answer'].upper()} ({result['confidence']:.1%})"); axes[1].axis("off")
fig.suptitle(f"Q: {QUESTION}", fontsize=12)
plt.tight_layout()
plt.savefig("results_vqa/inference_attention.png", dpi=150, bbox_inches="tight")
plt.show()